# Sprint 2 - Temporal Graph Construction (Elliptic)
**Inputs** (Google Drive, folder `Explainable-AML-GAT-ResNet/data`): `elliptic_features_scaled.csv`, `elliptic_edgelist_indexed.csv`, `txid_to_idx.json`, `elliptic_txs_classes.csv`, `elliptic_txs_features.csv` (only the txId and time-step columns are read from the last one).

**Outputs**: a `torch_geometric.data.Data` object (full graph + masks), a Train subgraph (time steps 1-34), a Test subgraph (time steps 35-49), and a list of per-time-step snapshots (discrete-time dynamic graph). Everything is saved to `elliptic_graph_temporal.pt`.

Label convention: `1 = illicit`, `0 = licit`, `-1 = unknown`.

In [1]:
!pip -q install torch_geometric
import os, json, glob
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from torch_geometric.utils import subgraph, contains_self_loops
print('torch', torch.__version__)

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


torch 2.14.1+cpu


## 1. Mount Drive and configure paths
> The notebook first looks for the files in My Drive. If they are not there, it authenticates with your Google account and downloads them from the *Shared with me* folder `Explainable-AML-GAT-ResNet/data` into the Colab disk (about 1 GB, a few minutes). Results are saved to `My Drive/Explainable-AML-GAT-ResNet_outputs`.

In [2]:
import io
from google.colab import drive, auth
drive.mount('/content/drive')

REQUIRED = ['txid_to_idx.json', 'elliptic_features_scaled.csv', 'elliptic_edgelist_indexed.csv',
            'elliptic_txs_classes.csv', 'elliptic_txs_features.csv']

DATA_DIR  = None   # set manually to skip auto-detection
FOLDER_ID = None   # optional: ID of the shared 'data' folder (the part after /folders/ in its URL)
LOCAL_DIR = '/content/elliptic_data'                                   # where downloaded files go
OUT_DIR   = '/content/drive/MyDrive/Explainable-AML-GAT-ResNet_outputs'  # where results are saved
os.makedirs(OUT_DIR, exist_ok=True)

def has_all(d):
    return os.path.isdir(d) and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

# Option A: the files are already inside My Drive (own copy or a working shortcut)
if DATA_DIR is None:
    for d in ['/content/drive/MyDrive/Explainable-AML-GAT-ResNet/data',
              '/content/drive/MyDrive/data', '/content/drive/MyDrive']:
        if has_all(d):
            DATA_DIR = d
            break

# Option B: download from the 'Shared with me' folder through the Drive API (no shortcut or copy needed)
if DATA_DIR is None:
    auth.authenticate_user()
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    svc = build('drive', 'v3', cache_discovery=False)
    KW = dict(supportsAllDrives=True, includeItemsFromAllDrives=True)

    def list_items(q):
        items, token = [], None
        while True:
            r = svc.files().list(q=q, fields='nextPageToken, files(id, name, size)',
                                 pageToken=token, pageSize=200, **KW).execute()
            items += r['files']
            token = r.get('nextPageToken')
            if not token:
                return items

    folder_id = FOLDER_ID
    if folder_id is None:
        FOLDER_MIME = "mimeType='application/vnd.google-apps.folder'"
        for p in list_items(f"{FOLDER_MIME} and name='Explainable-AML-GAT-ResNet' and trashed=false"):
            sub = list_items(f"'{p['id']}' in parents and {FOLDER_MIME} and name='data' and trashed=false")
            if sub:
                folder_id = sub[0]['id']
                break
    assert folder_id, 'Shared folder Explainable-AML-GAT-ResNet/data not found. Set FOLDER_ID manually.'

    remote = {f['name']: f for f in list_items(f"'{folder_id}' in parents and trashed=false")}
    missing = [n for n in REQUIRED if n not in remote]
    assert not missing, f'Missing files in the shared folder: {missing}'

    os.makedirs(LOCAL_DIR, exist_ok=True)
    for name in REQUIRED:
        f, dst = remote[name], os.path.join(LOCAL_DIR, name)
        if os.path.exists(dst) and os.path.getsize(dst) == int(f.get('size', -1)):
            print('cached    ', name)
            continue
        req = svc.files().get_media(fileId=f['id'], supportsAllDrives=True)
        with open(dst, 'wb') as fh:
            dl, done = MediaIoBaseDownload(fh, req, chunksize=64 * 1024 * 1024), False
            while not done:
                _, done = dl.next_chunk()
        print(f'downloaded {name} ({os.path.getsize(dst)/1e6:.1f} MB)')
    DATA_DIR = LOCAL_DIR

print('DATA_DIR =', DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

P = lambda name: os.path.join(DATA_DIR, name)
TRAIN_RANGE = (1, 34)
TEST_RANGE  = (35, 49)

Mounted at /content/drive


downloaded txid_to_idx.json (4.1 MB)
downloaded elliptic_features_scaled.csv (396.4 MB)
downloaded elliptic_edgelist_indexed.csv (3.0 MB)
downloaded elliptic_txs_classes.csv (3.3 MB)
downloaded elliptic_txs_features.csv (689.7 MB)
DATA_DIR = /content/elliptic_data
['elliptic_edgelist_indexed.csv', 'elliptic_features_scaled.csv', 'elliptic_txs_classes.csv', 'elliptic_txs_features.csv', 'txid_to_idx.json']


## 2. Metadata: txId -> idx, time step, labels

In [3]:
with open(P('txid_to_idx.json')) as f:
    txid_to_idx = {int(k): int(v) for k, v in json.load(f).items()}
N = len(txid_to_idx)

# Read only the first two columns (txId, time step) of the raw features file
meta = pd.read_csv(P('elliptic_txs_features.csv'), header=None, usecols=[0, 1])
meta.columns = ['txId', 'time_step']
meta['idx'] = meta['txId'].map(txid_to_idx)
assert meta['idx'].notna().all(), 'Some txIds are missing from txid_to_idx.json'
meta['idx'] = meta['idx'].astype(np.int64)
assert len(meta) == N and meta['idx'].is_unique and meta['idx'].max() == N - 1

time_step = torch.zeros(N, dtype=torch.long)
time_step[torch.tensor(meta['idx'].values)] = torch.tensor(meta['time_step'].values)

cls = pd.read_csv(P('elliptic_txs_classes.csv'))
cls.columns = ['txId', 'class']
cls['y'] = cls['class'].astype(str).str.strip().map({'unknown': -1, '1': 1, '2': 0})
assert cls['y'].notna().all(), 'Unexpected class values in elliptic_txs_classes.csv'
cls['idx'] = cls['txId'].map(txid_to_idx)
y = torch.full((N,), -1, dtype=torch.long)
y[torch.tensor(cls['idx'].values)] = torch.tensor(cls['y'].values.astype(np.int64))

print('Number of nodes:', N, '| time steps:', int(time_step.min()), '->', int(time_step.max()))
print('Labels (illicit=1, licit=0, unknown=-1):', {int(k): int(v) for k, v in zip(*torch.unique(y, return_counts=True))})

Number of nodes: 203769 | time steps: 1 -> 49
Labels (illicit=1, licit=0, unknown=-1): {-1: 157205, 0: 42019, 1: 4545}


## 3. Scaled feature matrix (from Sprint 1, exactly 166 features)

In [4]:
df = pd.read_csv(P('elliptic_features_scaled.csv'))   # new file: header row, exactly 166 feature columns, no id/label columns
print('Columns in CSV:', df.shape[1])
assert df.shape == (N, 166), (
    f'elliptic_features_scaled.csv has shape {df.shape}, expected ({N}, 166). '
    'Probably an outdated file: upload the new version to the data folder and re-run.')

x_rows = torch.tensor(df.to_numpy(dtype=np.float32))
print('Feature matrix:', tuple(x_rows.shape), '(expected 166 columns)')

# Row i of the file matches row i of the raw file -> place it at its mapped node index
x = torch.zeros_like(x_rows)
x[torch.tensor(meta['idx'].values)] = x_rows
del df, x_rows
print('NaN values in x:', bool(torch.isnan(x).any()))

Columns in CSV: 166
Feature matrix: (203769, 166) (expected 166 columns)
NaN values in x: False


## 4. Edge index (`[2, E]`, dtype `long`, directed)

In [5]:
probe = pd.read_csv(P('elliptic_edgelist_indexed.csv'), nrows=2, header=None)
has_header = not pd.to_numeric(probe.iloc[0], errors='coerce').notna().all()
e = pd.read_csv(P('elliptic_edgelist_indexed.csv'), header=0 if has_header else None).iloc[:, :2]
edge_index = torch.tensor(e.to_numpy(dtype=np.int64).T, dtype=torch.long).contiguous()

# If the file still contains raw txIds, map them to node indices
if int(edge_index.max()) >= N:
    edge_index = torch.tensor(e.apply(lambda c: c.map(txid_to_idx)).to_numpy(dtype=np.int64).T).contiguous()

assert edge_index.dim() == 2 and edge_index.size(0) == 2
assert 0 <= int(edge_index.min()) and int(edge_index.max()) < N
cross = int((time_step[edge_index[0]] != time_step[edge_index[1]]).sum())
print('edge_index:', tuple(edge_index.shape), edge_index.dtype)
print('Has self-loops:', contains_self_loops(edge_index), '| edges across different time steps:', cross)

edge_index: (2, 234355) torch.int64
Has self-loops: False | edges across different time steps: 0


## 5. Full `Data` object with temporal masks

In [6]:
train_mask = (time_step >= TRAIN_RANGE[0]) & (time_step <= TRAIN_RANGE[1])
test_mask  = (time_step >= TEST_RANGE[0])  & (time_step <= TEST_RANGE[1])
labeled = y != -1

data = Data(x=x, edge_index=edge_index, y=y, time_step=time_step,
            train_mask=train_mask, test_mask=test_mask,
            train_labeled_mask=train_mask & labeled,
            test_labeled_mask=test_mask & labeled)
data.validate(raise_on_error=True)
print(data)
print('Train nodes:', int(train_mask.sum()), '(labeled:', int(data.train_labeled_mask.sum()), ')')
print('Test  nodes:', int(test_mask.sum()),  '(labeled:', int(data.test_labeled_mask.sum()),  ')')

Data(x=[203769, 166], edge_index=[2, 234355], y=[203769], time_step=[203769], train_mask=[203769], test_mask=[203769], train_labeled_mask=[203769], test_labeled_mask=[203769])
Train nodes: 136265 (labeled: 29894 )
Test  nodes: 67504 (labeled: 16670 )


## 6. Train (t=1-34) / Test (t=35-49) subgraphs and per-time-step snapshots

In [7]:
def make_subgraph(node_mask):
    ei, _ = subgraph(node_mask, edge_index, relabel_nodes=True, num_nodes=N)
    return Data(x=x[node_mask], edge_index=ei, y=y[node_mask], time_step=time_step[node_mask],
                orig_idx=torch.nonzero(node_mask).view(-1))

train_data = make_subgraph(train_mask)
test_data  = make_subgraph(test_mask)

# Discrete-time dynamic graph: one snapshot per time step
snapshots = [make_subgraph(time_step == t) for t in range(1, 50)]

print('Train:', train_data)
print('Test :', test_data)
assert train_data.edge_index.max() < train_data.num_nodes
assert test_data.edge_index.max() < test_data.num_nodes
assert int(train_data.time_step.max()) <= TRAIN_RANGE[1] and int(test_data.time_step.min()) >= TEST_RANGE[0]

summary = pd.DataFrame({
    'time_step': range(1, 50),
    'nodes': [s.num_nodes for s in snapshots],
    'edges': [s.num_edges for s in snapshots],
    'illicit': [int((s.y == 1).sum()) for s in snapshots],
    'licit': [int((s.y == 0).sum()) for s in snapshots],
    'unknown': [int((s.y == -1).sum()) for s in snapshots],
})
summary['split'] = np.where(summary.time_step <= TRAIN_RANGE[1], 'train', 'test')
display(summary)
print(summary.groupby('split')[['nodes', 'edges', 'illicit', 'licit', 'unknown']].sum())

Train: Data(x=[136265, 166], edge_index=[2, 156843], y=[136265], time_step=[136265], orig_idx=[136265])
Test : Data(x=[67504, 166], edge_index=[2, 77512], y=[67504], time_step=[67504], orig_idx=[67504])


,time_step,nodes,edges,illicit,licit,unknown,split
0,1,7880,9164,17,2130,5733,train
1,2,4544,5241,18,1099,3427,train
2,3,6621,8316,11,1268,5342,train
3,4,5693,8180,30,1410,4253,train
4,5,6803,8623,8,1874,4921,train
5,6,4328,5242,5,480,3843,train
6,7,6048,7253,102,1101,4845,train
7,8,4457,5186,67,1098,3292,train
8,9,4996,5939,248,530,4218,train
9,10,6727,8588,18,954,5755,train


        nodes   edges  illicit  licit  unknown
split                                         
test    67504   77512     1083  15587    50834
train  136265  156843     3462  26432   106371


## 7. Save results

In [8]:
from google.colab import files

OUT = '/content/elliptic_graph_temporal.pt'   # saved on the Colab disk (shows up in the Files panel)
torch.save({'data': data, 'train_data': train_data, 'test_data': test_data,
            'snapshots': snapshots, 'train_range': TRAIN_RANGE, 'test_range': TEST_RANGE}, OUT)
print('Saved:', OUT, f'({os.path.getsize(OUT)/1e6:.1f} MB)')

files.download(OUT)   # optional: sends the file to your browser's Downloads folder

# To reload (PyTorch >= 2.6 needs weights_only=False because the file contains PyG objects):
# ckpt = torch.load(OUT, weights_only=False); train_data = ckpt['train_data']

Saved: /content/elliptic_graph_temporal.pt (431.1 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>